# 01 — Data Collection

Gathers both corpora used later in this project:

1. **Wikitext-103** (via Hugging Face `datasets`) — the pretraining corpus.
2. **Real SEC filings** (10-K / 10-Q, 10 companies, filed 2022 onward) —
   scraped directly from [SEC EDGAR](https://www.sec.gov/edgar), used for
   BPE tokenizer training (`02_tokenizer.ipynb`) and fine-tuning
   (`04_finetune.ipynb`).

No RAG, chunking, or retrieval logic here — that's a separate concern
handled entirely in the application this model was later integrated into
(see the root README).

Originally run in Google Colab with Drive mounted at
`/content/drive/MyDrive/sec_chatbot/`; adjust `PROJECT_ROOT`-style paths if
running elsewhere.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q datasets transformers tokenizers beautifulsoup4 requests pandas

import os
import re
import json
import requests
import pandas as pd
from bs4 import BeautifulSoup
from datasets import load_dataset

# --- Download and save Wikitext-103 ---
print("Downloading Wikitext-103...")
dataset = load_dataset("Salesforce/wikitext", "wikitext-103-v1", split="train")
print(f"Rows downloaded: {len(dataset)}")

wiki_save_path = '/content/drive/MyDrive/sec_chatbot/data/wikitext103_train.txt'
with open(wiki_save_path, 'w', encoding='utf-8') as f:
    for row in dataset:
        text = row['text'].strip()
        if text:
            f.write(text + '\n')
print(f"Wikitext-103 saved to {wiki_save_path}")

# --- EDGAR setup ---
# EDGAR requires a User-Agent header identifying who you are
HEADERS = {'User-Agent': 'Aarya Kulkarni aarya.kmail@gmail.com'}
APPLE_CIK = '0000320193'

# --- Pull Apple's filing list as a first test ---
url = f'https://data.sec.gov/submissions/CIK{APPLE_CIK}.json'
response = requests.get(url, headers=HEADERS)
data = response.json()
print(f"Company: {data['name']}, Ticker: {data['tickers']}")

filings = data['filings']['recent']
df = pd.DataFrame({
    'form': filings['form'],
    'date': filings['filingDate'],
    'accession': filings['accessionNumber'],
    'primaryDoc': filings['primaryDocument']
})

df = df[df['form'].isin(['10-K', '10-Q'])]
df = df[df['date'] >= '2022-01-01']
print(f"\nFilings found: {len(df)}")
print(df[['form', 'date', 'primaryDoc']].head())

# --- Filing text extractor ---
def get_filing_text(cik, accession, primary_doc):
    acc_clean = accession.replace('-', '')
    cik_int = int(cik)
    doc_url = f'https://www.sec.gov/Archives/edgar/data/{cik_int}/{acc_clean}/{primary_doc}'

    resp = requests.get(doc_url, headers=HEADERS)
    soup = BeautifulSoup(resp.text, 'html.parser')

    if soup.head:
        soup.head.decompose()
    for tag in soup.find_all('div', style=lambda s: s and 'display:none' in s.replace(' ', '')):
        tag.decompose()
    for tag in soup(['script', 'style']):
        tag.decompose()

    text = soup.get_text(separator=' ', strip=True)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# --- Test on Apple's most recent filing ---
print("\nTesting extractor on Apple's most recent filing...")
test_text = get_filing_text(APPLE_CIK, df.iloc[0]['accession'], df.iloc[0]['primaryDoc'])
print(f"Characters extracted: {len(test_text)}")
print("\nFirst 500 characters:")
print(test_text[:500])

Mounted at /content/drive
Rows downloaded: 1801350
Wikitext-103 saved to /content/drive/MyDrive/sec_chatbot/data/wikitext103_train.txt
Company: Apple Inc., Ticker: ['AAPL']

Filings found: 18
    form        date         primaryDoc
9   10-Q  2026-05-01  aapl-20260328.htm
40  10-Q  2026-01-30  aapl-20251227.htm
51  10-K  2025-10-31  aapl-20250927.htm
77  10-Q  2025-08-01  aapl-20250628.htm
89  10-Q  2025-05-02  aapl-20250329.htm

Testing extractor on Apple's most recent filing...
Characters extracted: 84733

First 500 characters:
UNITED STATES SECURITIES AND EXCHANGE COMMISSION Washington, D.C. 20549 FORM 10-Q (Mark One) [X] QUARTERLY REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934 For the quarterly period ended March 28, 2026 or [ ] TRANSITION REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934 For the transition period from to . Commission File Number: 001-36743 Apple Inc. (Exact name of Registrant as specified in its charter) Ca

In [1]:
import time

# Folder to save Apple's filings
apple_save_dir = '/content/drive/MyDrive/sec_chatbot/data/apple'
os.makedirs(apple_save_dir, exist_ok=True)

for i, row in df.iterrows():
    filename = f"{row['date']}_{row['form'].replace('-', '')}_{row['primaryDoc'].replace('.htm', '')}.txt"
    save_path = os.path.join(apple_save_dir, filename)

    # Skip if already saved -- useful if Colab disconnects mid-loop
    if os.path.exists(save_path):
        print(f"Already saved, skipping: {filename}")
        continue

    print(f"Downloading: {row['date']} {row['form']}...")
    text = get_filing_text(APPLE_CIK, row['accession'], row['primaryDoc'])

    if text:
        with open(save_path, 'w', encoding='utf-8') as f:
            f.write(text)
        print(f"Saved: {filename} ({len(text)} characters)")
    else:
        print(f"WARNING: No text extracted for {filename}")

    # Wait 0.5s between requests so EDGAR doesn't rate-limit/block us
    time.sleep(0.5)

print("\nAll Apple filings downloaded.")

Downloading: 2026-05-01 10-Q...
Saved: 2026-05-01_10Q_aapl-20260328.txt (84733 characters)
...
Downloading: 2022-01-28 10-Q...
Saved: 2022-01-28_10Q_aapl-20211225.txt (58116 characters)

All Apple filings downloaded.

In [1]:
# CIK (Central Index Key) numbers for all 10 companies
COMPANIES = {
    'apple':      '0000320193',
    'microsoft':  '0000789019',
    'nvidia':     '0001045810',
    'alphabet':   '0001652044',
    'amazon':     '0001018724',
    'meta':       '0001326801',
    'broadcom':   '0001730168',
    'tesla':      '0001318605',
    'oracle':     '0001341439',
    'salesforce': '0001108524',
}

for company_name, cik in COMPANIES.items():
    print(f"\n{'='*40}\nProcessing: {company_name.upper()}\n{'='*40}")

    url = f'https://data.sec.gov/submissions/CIK{cik}.json'
    resp = requests.get(url, headers=HEADERS)
    company_data = resp.json()

    filings = company_data['filings']['recent']
    company_df = pd.DataFrame({
        'form':       filings['form'],
        'date':       filings['filingDate'],
        'accession':  filings['accessionNumber'],
        'primaryDoc': filings['primaryDocument']
    })

    company_df = company_df[company_df['form'].isin(['10-K', '10-Q'])]
    company_df = company_df[company_df['date'] >= '2022-01-01']
    print(f"Filings found: {len(company_df)}")

    save_dir = f'/content/drive/MyDrive/sec_chatbot/data/{company_name}'
    os.makedirs(save_dir, exist_ok=True)

    for i, row in company_df.iterrows():
        filename = f"{row['date']}_{row['form'].replace('-', '')}_{row['primaryDoc'].replace('.htm', '')}.txt"
        save_path = os.path.join(save_dir, filename)

        if os.path.exists(save_path):
            print(f"  Already saved, skipping: {filename}")
            continue

        print(f"  Downloading: {row['date']} {row['form']}...")
        text = get_filing_text(cik, row['accession'], row['primaryDoc'])

        if text:
            with open(save_path, 'w', encoding='utf-8') as f:
                f.write(text)
            print(f"  Saved: {filename} ({len(text)} chars)")
        else:
            print(f"  WARNING: No text for {filename}")

        time.sleep(0.5)

print("\n\nAll companies done.")

Processing: SALESFORCE
Filings found: 12
  Downloading: 2026-05-28 10-Q...
  Saved: 2026-05-28_10Q_crm-20260430.txt (287963 chars)
  ...
  Downloading: 2023-08-31 10-Q...
  Saved: 2023-08-31_10Q_crm-20230731.txt (292089 chars)

All companies done.

In [1]:
import os

data_dir = '/content/drive/MyDrive/sec_chatbot/data'
total_files = 0
total_chars = 0

for company in os.listdir(data_dir):
    company_path = os.path.join(data_dir, company)
    if os.path.isdir(company_path):
        files = [f for f in os.listdir(company_path) if f.endswith('.txt')]
        chars = sum(os.path.getsize(os.path.join(company_path, f)) for f in files)
        print(f"{company:15} {len(files):3} filings  {chars/1_000_000:.1f} MB")
        total_files += len(files)
        total_chars += chars

print(f"\nTotal: {total_files} files, {total_chars/1_000_000:.1f} MB")

apple            18 filings  1.7 MB
microsoft        18 filings  4.2 MB
nvidia           18 filings  3.5 MB
alphabet         12 filings  2.4 MB
amazon           18 filings  3.6 MB
meta              8 filings  3.2 MB
broadcom         18 filings  4.4 MB
tesla            18 filings  3.9 MB
oracle           18 filings  4.0 MB
salesforce       12 filings  3.8 MB

Total: 158 files, 34.8 MB